# PySpark interview challenge

**Time limit:** 60 minutes. Use the empty cells under each task. Don't open `interview_challenge_solutions.md` until you're done.

**Assign every result to the variable named in the task.** The check cell after each task reads that variable. A check that fails and says `NameError` means the variable is missing.

**Context:** an e-commerce company wants a data analyst to answer questions about orders, customers, products and web traffic. The data is in `/data`:

| File | Contents |
|---|---|
| `customers.csv` | customer_id, name (`customer_00001`), city, state, segment, signup_date |
| `products.csv` | product_id, product_name, category, unit_price |
| `orders.csv` | order_id, customer_id, product_id, quantity, order_ts, status |
| `web_events.jsonl` | customer_id, product_id, event_type (view, add_to_cart, purchase), event_ts |

Only orders with `status = 'delivered'` count as revenue. Revenue of an order = `quantity * unit_price`.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
import contextlib, io
import os

base_path = os.path.dirname(os.getcwd())

spark = (
    SparkSession.builder
    .appName("interview-challenge")
    .master("spark://localhost:7077")
    .config("spark.driver.host", "172.24.0.1")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)

# Check helper: runs a check, prints PASS/FAIL, and never stops the notebook
results = []

def check(name, fn):
    try:
        ok, detail = fn()
    except Exception as e:
        ok, detail = False, f"{type(e).__name__}: {str(e)[:150]}"
    results.append((name, bool(ok)))
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] {name}" + (f" - {detail}" if detail else ""))

def summary():
    passed = sum(ok for _, ok in results)
    print(f"\n{passed}/{len(results)} checks passed")

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/04 09:30:38 WARN Utils: Your hostname, <hostname>, resolves to a loopback address: 127.0.1.1; using <host-ip> instead (on interface wlo1)
26/11/04 09:30:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 09:30:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.0


## Task 1 — Load the data with an explicit schema (10 min)

Load all four files into DataFrames named `customers`, `products`, `orders` and `events`. Don't use `inferSchema`. Define each schema yourself, with the right types for dates and timestamps.

Print each schema.

In [2]:
products = spark.read.csv(os.path.join(base_path, "data", "products.csv"), header=True, inferSchema=True)
orders = spark.read.csv(os.path.join(base_path, "data", "orders.csv"), header=True, inferSchema=True)
events = spark.read.json(os.path.join(base_path, "data", "web_events.jsonl"), multiLine=True)


In [3]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("city", StringType(), False),
    StructField("state", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("signup_date", DateType(), False),
])

customers = spark.read.csv(
    os.path.join(base_path, "data", "customers.csv"),
    header=True,
    schema=customers_schema
)

In [4]:
check("Task 1: customers.signup_date is date", lambda: (dict(customers.dtypes)["signup_date"] == "date", ""))
check("Task 1: orders.order_ts is timestamp", lambda: (dict(orders.dtypes)["order_ts"] == "timestamp", ""))
check("Task 1: products.unit_price is double", lambda: (dict(products.dtypes)["unit_price"] == "double", ""))
check("Task 1: events has event_type", lambda: ("event_type" in events.columns, ""))

[PASS] Task 1: customers.signup_date is date
[PASS] Task 1: orders.order_ts is timestamp
[PASS] Task 1: products.unit_price is double
[PASS] Task 1: events has event_type


## Task 2 — Top 5 customers by revenue (10 min)

Assign the result to `top_customers`: the 5 customers with the highest delivered revenue. Columns: `customer_id`, `name`, `city`, `revenue` (rounded to 2 decimals) and `order_count`. Order by revenue, highest first.

In [5]:
top_customers = (
    orders.filter(F.col("status") == "delivered").alias("o")
    .join(products.alias("p"), F.col("o.product_id") == F.col("p.product_id"))
    .join(customers.alias("c"), F.col("o.customer_id") == F.col("c.customer_id"))
    .groupBy(F.col("c.customer_id"), F.col("c.name"), F.col("c.city"))
    .agg(
        F.round(F.sum(F.col("o.quantity") * F.col("p.unit_price")), 2).alias("revenue"),
        F.countDistinct("o.order_id").alias("order_count")
    )
    .orderBy(F.desc("revenue"))
    .limit(5)
)
top_customers.show()

+-----------+--------------+--------------+--------+-----------+
|customer_id|          name|          city| revenue|order_count|
+-----------+--------------+--------------+--------+-----------+
|          1|customer_00001|Rio de Janeiro|44564.11|          7|
|          9|customer_00009|      Brasília|22302.15|          2|
|         10|customer_00010|     Fortaleza|15879.02|          3|
|         55|customer_00055|      Salvador| 12042.5|          2|
|          3|customer_00003|Belo Horizonte| 6434.68|          4|
+-----------+--------------+--------------+--------+-----------+



In [6]:
def reference_top5_revenue():
    orders.createOrReplaceTempView("orders_v")
    products.createOrReplaceTempView("products_v")
    return spark.sql("""
        SELECT ROUND(SUM(o.quantity * p.unit_price), 2) AS revenue
        FROM orders_v o JOIN products_v p ON o.product_id = p.product_id
        WHERE o.status = 'delivered'
        GROUP BY o.customer_id
        ORDER BY revenue DESC
        LIMIT 5
    """).collect()

check("Task 2: top_customers has 5 rows", lambda: (top_customers.count() == 5, ""))
check("Task 2: expected columns", lambda: (
    {"customer_id", "name", "city", "revenue", "order_count"}.issubset(top_customers.columns), ""))
check("Task 2: revenue values match an independent SQL query", lambda: (
    [round(r.revenue, 2) for r in top_customers.orderBy(F.desc("revenue")).collect()]
    == [round(r.revenue, 2) for r in reference_top5_revenue()], ""))

[PASS] Task 2: top_customers has 5 rows
[PASS] Task 2: expected columns


[PASS] Task 2: revenue values match an independent SQL query


## Task 3 — Monthly revenue in 2024 (5 min)

Assign to `monthly_revenue`: delivered revenue per month in 2024, with a `month` column in the format `yyyy-MM` and a `revenue` column. Order by month.

In [7]:
monthly_revenue = orders.alias('o')\
    .filter(F.col('status') == 'delivered')\
    .withColumn('year', F.year(F.col('o.order_ts')))\
    .filter(F.col('year') == 2024)\
    .withColumn('month', F.date_format(orders.order_ts, 'yyyy-MM'))\
    .join(products.alias('p'), F.col('o.product_id') == F.col('p.product_id'))\
    .withColumn('revenue', F.round(F.col('quantity') * F.col('unit_price')))\
    .select(['month', 'revenue'])\
    .orderBy('month', ascendente=True)

In [8]:
reference_row = spark.sql("""
    SELECT SUM(o.quantity * p.unit_price)
    FROM orders_v o JOIN products_v p ON o.product_id = p.product_id
    WHERE o.status = 'delivered' AND YEAR(o.order_ts) = 2024
""").first()

reference_2024_total = (
    float(reference_row[0])
    if reference_row is not None and reference_row[0] is not None
    else 0.0
)

check("Task 3: every month is in 2024", lambda: (
    all(r.month.startswith("2024-") for r in monthly_revenue.collect()), ""))
check("Task 3: months are ordered", lambda: (
    [r.month for r in monthly_revenue.collect()] == sorted(r.month for r in monthly_revenue.collect()), ""))
check("Task 3: total matches the 2024 reference", lambda: (
    abs(monthly_revenue.agg(F.sum("revenue")).first()[0] - reference_2024_total) < 1.0,
    f"reference {reference_2024_total:.2f}"))

[PASS] Task 3: every month is in 2024
[PASS] Task 3: months are ordered
[PASS] Task 3: total matches the 2024 reference - reference 67033.14


## Task 4 — Customers who never bought (5 min)

Assign to `never_bought`: customers with no orders in any status. Columns `customer_id`, `name` and `segment`.

In [21]:
customers.createOrReplaceTempView('customers_v')
orders.createOrReplaceTempView('orders_v')

In [40]:
never_bought = spark.sql("""
    SELECT c.customer_id, c.name, c.segment
    FROM customers_v AS c
    LEFT JOIN orders_v AS o
      ON c.customer_id = o.customer_id
    WHERE o.order_id is null
""")
never_bought.show()

+-----------+--------------+--------------+
|customer_id|          name|       segment|
+-----------+--------------+--------------+
|         13|customer_00013|small_business|
|         15|customer_00015|small_business|
|         16|customer_00016|      consumer|
|         17|customer_00017|      consumer|
|         21|customer_00021|    enterprise|
|         25|customer_00025|small_business|
|         27|customer_00027|small_business|
|         30|customer_00030|    enterprise|
|         35|customer_00035|      consumer|
|         36|customer_00036|small_business|
|         39|customer_00039|    enterprise|
|         40|customer_00040|small_business|
|         41|customer_00041|    enterprise|
|         44|customer_00044|      consumer|
|         46|customer_00046|      consumer|
|         48|customer_00048|    enterprise|
|         50|customer_00050|    enterprise|
|         51|customer_00051|small_business|
|         52|customer_00052|      consumer|
|         58|customer_00058|    

In [41]:

check("Task 4: never_bought has the expected columns", lambda: (
    {"customer_id", "name", "segment"}.issubset(never_bought.columns), ""))
check("Task 4: row count matches the anti-join reference", lambda: (
    never_bought.count() == expected_never, f"got {never_bought.count()}, expected {expected_never}"))
check("Task 4: no returned customer has an order", lambda: (
    never_bought.join(orders, "customer_id").count() == 0, ""))

[PASS] Task 4: never_bought has the expected columns
[PASS] Task 4: row count matches the anti-join reference - got 47, expected 47
[PASS] Task 4: no returned customer has an order


## Task 5 — Parse the customer number (5 min)

Reassign `customers` with a new integer column `customer_number` taken from `name` (`customer_00042` becomes `42`). Then assign to `div_by_100` the number of customers whose `customer_number` is divisible by 100.

In [ ]:
customers = customers.withColumn(
    "customer_number",
    F.regexp_extract(F.col("name"), r"_(\d+)$", 1).cast("int")
)

div_by_100 = customers.filter(
    F.col("customer_number") % 100 == 0
).count()

TypeError: cast() missing 1 required positional argument: 'val'

In [12]:
expected_div_by_100 = sum(
    1 for r in customers.select("name").collect() if int(r.name.split("_")[1]) % 100 == 0
)

check("Task 5: customers has customer_number", lambda: ("customer_number" in customers.columns, ""))
check("Task 5: customer_number is integer", lambda: (dict(customers.dtypes)["customer_number"] == "int", ""))
check("Task 5: div_by_100 matches the Python reference", lambda: (
    div_by_100 == expected_div_by_100, f"got {div_by_100}, expected {expected_div_by_100}"))

[FAIL] Task 5: customers has customer_number
[FAIL] Task 5: customer_number is integer - KeyError: 'customer_number'
[FAIL] Task 5: div_by_100 matches the Python reference - NameError: name 'div_by_100' is not defined


## Task 6 — Conversion rate per product (10 min)

Assign to `conversion` the top 5 products by conversion rate (`purchases / views`), among products with at least 10 views. Columns: `product_name`, `views`, `purchases`, `conversion_rate` (rounded to 3 decimals). Order by conversion rate, highest first.

In [13]:
# your code here


In [14]:
events.createOrReplaceTempView("events_v")
reference_rates = [round(r.rate, 3) for r in spark.sql("""
    SELECT SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) / SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS rate
    FROM events_v
    GROUP BY product_id
    HAVING SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) >= 10
    ORDER BY rate DESC
    LIMIT 5
""").collect()]

check("Task 6: at most 5 rows", lambda: (conversion.count() <= 5, ""))
check("Task 6: every product has at least 10 views", lambda: (
    conversion.filter(F.col("views") < 10).count() == 0, ""))
check("Task 6: sorted by conversion rate, highest first", lambda: (
    [r.conversion_rate for r in conversion.collect()]
    == sorted([r.conversion_rate for r in conversion.collect()], reverse=True), ""))
check("Task 6: rates match an independent SQL query", lambda: (
    [round(r.conversion_rate, 3) for r in conversion.collect()] == reference_rates,
    f"expected {reference_rates}"))

[FAIL] Task 6: at most 5 rows - NameError: name 'conversion' is not defined
[FAIL] Task 6: every product has at least 10 views - NameError: name 'conversion' is not defined
[FAIL] Task 6: sorted by conversion rate, highest first - NameError: name 'conversion' is not defined
[FAIL] Task 6: rates match an independent SQL query - NameError: name 'conversion' is not defined


## Task 7 — Best product per category (10 min)

Assign to `best_per_category`: for each category, the product with the highest delivered revenue. Columns `category`, `product_name`, `revenue`. Use a window function, not a self-join.

In [15]:
# your code here


In [16]:
reference_best = spark.sql("""
    SELECT category, revenue FROM (
        SELECT p.category, ROUND(SUM(o.quantity * p.unit_price), 2) AS revenue,
               ROW_NUMBER() OVER (PARTITION BY p.category ORDER BY SUM(o.quantity * p.unit_price) DESC) AS rn
        FROM orders_v o JOIN products_v p ON o.product_id = p.product_id
        WHERE o.status = 'delivered'
        GROUP BY p.category, o.product_id
    ) WHERE rn = 1
""").collect()
reference_pairs = {(r.category, round(r.revenue, 2)) for r in reference_best}

check("Task 7: one row per category", lambda: (
    best_per_category.count() == best_per_category.select("category").distinct().count(), ""))
check("Task 7: best revenue per category matches an independent SQL query", lambda: (
    {(r.category, round(r.revenue, 2)) for r in best_per_category.collect()} == reference_pairs, ""))

[FAIL] Task 7: one row per category - NameError: name 'best_per_category' is not defined
[FAIL] Task 7: best revenue per category matches an independent SQL query - NameError: name 'best_per_category' is not defined


## Task 8 — Performance question (5 min)

Assign to `broadcast_joined` the join of `orders` and `products` with `products` broadcast. Print its plan with `.explain()`. Then answer the questions in the markdown cell below.

1. Which join strategy does Spark use by default for this join?
2. How does the broadcast change the plan, and why?
3. Why can a broadcast join fail on a large table, and what setting controls it?

In [17]:
# your code here


In [18]:
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    broadcast_joined.explain()

check("Task 8: plan uses BroadcastHashJoin", lambda: ("BroadcastHashJoin" in buf.getvalue(), ""))
check("Task 8: plan has no Exchange on the orders side", lambda: (
    buf.getvalue().count("Exchange") <= 1, f"Exchange count {buf.getvalue().count('Exchange')}"))

NameError: name 'broadcast_joined' is not defined

*Your answers:*

1. 
2. 
3. 

In [ ]:
summary()